# Notebook Colab — SIMONE (Transcription & Traduction)

Ce notebook prépare l'environnement sur Google Colab, permet d'uploader ou cloner le projet, installe les dépendances, puis exécute la pipeline de transcription + traduction sur un fichier audio.

**Remarque** : l'accès micro en temps réel n'est pas natif sur Colab ; utilisez l'upload de fichiers audio ou l'enregistreur JS fourni plus bas (optionnel).

In [ ]:
# 1) Installer ffmpeg système et dépendances Python (exécuter une seule fois)
!apt-get update -y && apt-get install -y ffmpeg
# Installer paquets Python principaux (adaptés au projet)
!pip install -q -U openai-whisper transformers sentencepiece imageio-ffmpeg python-docx reportlab
# Optionnel: faster-whisper pour de meilleures performances (GPU/CPU)
# !pip install -q faster-whisper

In [ ]:
# 2) Monter Google Drive (optionnel mais recommandé pour stocker modèles/outputs)
from google.colab import drive
drive.mount('/content/drive')

## 3) Récupérer le code du projet
Deux options : cloner depuis GitHub (si tu as un repo) ou uploader manuellement les fichiers du projet (zip ou dossier).

In [ ]:
# Option A : cloner depuis GitHub (décommente et modifie l'URL si nécessaire)
# !git clone https://github.com/TON_COMPTE/TON_REPO.git simone_repo
# %cd simone_repo

# Option B : uploader manuellement via l'UI Colab
from google.colab import files
print('Sélectionner les fichiers du projet (ou un zip)')
uploaded = files.upload()
if uploaded:
  for fn in uploaded:
    print('Uploaded', fn)
# Si vous avez uploadé un zip, décommentez et extrayez le ci-dessous si nécessaire
# import zipfile, os
# for fn in uploaded:
#   if fn.endswith('.zip'):
#       with zipfile.ZipFile(fn, 'r') as z:
#           z.extractall('.')
#       print('extracted', fn)

## 4) Installer les dépendances du projet (si `requirements.txt` est présent)
Exécutez la cellule suivante depuis le répertoire racine du projet (après git clone ou upload).

In [ ]:
# Installer les dépendances du projet si le fichier requirements.txt est présent
import os
if os.path.exists('requirements.txt'):
  !pip install -r requirements.txt
else:
  print('requirements.txt non trouvé dans le répertoire courant. Assurez-vous dêtre dans le dossier du projet.')

## 5) Choisir et forcer le modèle Whisper pour Colab
Pour Colab GPU: `small` ou `medium` sont de bons compromis. Pour CPU ou exécutions rapides, `tiny`.
Définissez la variable d'environnement `WHISPER_MODEL` avant d'exécuter le script.

In [ ]:
# Exemple : définir le modèle Whisper à utiliser
import os
os.environ['WHISPER_MODEL'] = 'small'  # ou 'tiny', 'medium' selon ton GPU
print('WHISPER_MODEL =', os.environ['WHISPER_MODEL'])

## 6) Exécuter la pipeline sur un fichier audio (exemple)
Upload un fichier audio (wav/mp3) via l'UI puis exécutez la cellule suivante en remplaçant `audio_file_path`.

In [ ]:
# Exemple : uploader un fichier audio et lancer la transcription+traduction via le code du projet
from google.colab import files
uploaded = files.upload()
audio_path = None
for fn in uploaded:
  audio_path = fn
  print('Fichier audio chargé:', fn)

# Ajouter le dossier du projet au sys.path si nécessaire
import sys, os
proj_dir = os.getcwd()  # adapter si vous avez cloné dans un sous-dossier
if proj_dir not in sys.path:
  sys.path.append(proj_dir)

# Importer et exécuter la pipeline (assure-toi que le projet est dans le répertoire courant)
try:
  from backend.pipeline import audio_to_translated_text
  res = audio_to_translated_text(audio_path=audio_path, source_lang=None, target_lang='fr')
  print('=== ORIGINAL ===')
  print(res.get('original_text'))
  print('
=== DETECTED LANGUAGE ===')
  print(res.get('detected_source'))
  print('
=== TRANSLATED ===')
  print(res.get('translated_text'))
except Exception as e:
  print('Erreur lors de l exécution de la pipeline :', e)

## 7) (Optionnel) Enregistreur micro côté navigateur
Colab n'a pas un accès direct au micro du runtime. La cellule suivante propose un petit enregistreur JavaScript côté client.
Après enregistrement, télécharge le fichier .wav généré puis exécute la cellule d'upload ci-dessus pour le traiter.

In [ ]:
from IPython.display import Javascript, display
display(Javascript("""
(function(){
  const sleep = t => new Promise(r => setTimeout(r, t));
  async function record(){
    const stream = await navigator.mediaDevices.getUserMedia({audio:true});
    const mediaRecorder = new MediaRecorder(stream);
    const chunks = [];
    mediaRecorder.ondataavailable = e => chunks.push(e.data);
    mediaRecorder.start();
    const recButton = document.createElement('button');
    recButton.textContent = 'Stop and download';
    document.body.appendChild(recButton);
    recButton.addEventListener('click', async () => {
      mediaRecorder.stop();
      await sleep(500);
      const blob = new Blob(chunks, {type:'audio/wav'});
      const url = URL.createObjectURL(blob);
      const a = document.createElement('a');
      a.href = url;
      a.download = 'recording.wav';
      a.textContent = 'Download recording.wav';
      document.body.appendChild(a);
    });
  }
  record();
})();
"""))

## 8) Notes finales et bonnes pratiques
- Pour Colab gratuit, privilégiez `small` ou `tiny`. Pour Colab Pro, `medium` peut être utilisable selon la RAM du runtime.
- Monte Drive pour conserver les modèles et les outputs.
- Si tu veux que je prépare un notebook personnalisé qui clone directement ton repo GitHub privé (avec PAT) et précharge un modèle particulier, dis‑le moi.